# Exploración de los datos de BiciBCN

Usad este notebook para conocer los datos **antes** de escribir `SPECS.md`. Lo que encontréis va a la sección **Entradas** de `SPECS.md`, y las decisiones a **Decisiones**.

Trabajamos con pandas. Para ficheros grandes, polars es más rápido (`pl.read_csv`, misma lógica).

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

DATOS = Path("../data/raw")
tablas = {n: pd.read_csv(DATOS / f"{n}.csv") for n in ['estaciones', 'barrios', 'distritos', 'viajes', 'estado_estaciones', 'redistribucion', 'abonados', 'meteo', 'festivos']}
for nombre, df in tablas.items():
    print(f"{nombre:20s} {df.shape[0]:>10,d} filas  {df.shape[1]:>2d} columnas")

estaciones                  543 filas   8 columnas
barrios                      73 filas   4 columnas
distritos                    10 filas   4 columnas
viajes                  774,981 filas   8 columnas
estado_estaciones     1,185,912 filas   6 columnas
redistribucion           29,162 filas   6 columnas
abonados                168,000 filas   6 columnas
meteo                       365 filas   4 columnas
festivos                     13 filas   2 columnas


## 1. Perfil de cada tabla

In [3]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    """Tipo, nulos, valores distintos y un ejemplo de cada columna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(1),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0],
    })

perfil(tablas['viajes'])

,tipo,nulos,% nulos,distintos,ejemplo
viaje_id,str,0,0.0,771893,V0000001
abonado_id,str,0,0.0,136266,A050021
bici_id,str,0,0.0,8000,E1095
tipo_bici,str,0,0.0,4,ELECTRIC
estacion_origen,int64,0,0.0,540,176
fecha_hora_inicio,str,0,0.0,744805,2025-07-01 00:11:23
estacion_destino,float64,86,0.0,542,162.0
fecha_hora_fin,str,86,0.0,757795,2025-07-01 00:14:21


## 2. Preguntas de calidad para empezar

- ¿Es `viaje_id` único?
- ¿Qué valores toma `tipo_bici`?
- ¿Cómo se distribuye la duración de los viajes? ¿Hay negativos, nulos o viajes de días?
- ¿Todas las estaciones de `viajes` existen en `estaciones`?
- ¿Todas las estaciones están en Barcelona? ¿Qué altitudes y capacidades hay?
- ¿Coincide el distrito de cada estación con el de su barrio?
- En `estado_estaciones`: ¿puede haber más bicis que anclajes? ¿Hay estaciones que no cambian nunca?
- ¿Hay días sin viajes?

In [13]:
districte_per_barri = tablas["barrios"].set_index("codi_barri")["codi_districte"]

comparacion = estaciones[
    ["estacion_id", "nombre", "codi_barri", "codi_districte"]
].copy()
comparacion["districte_del_barri"] = comparacion["codi_barri"].map(districte_per_barri)

sin_barrio = comparacion["districte_del_barri"].isna()
discrepancias = (
    ~sin_barrio
    & comparacion["codi_districte"].ne(comparacion["districte_del_barri"])
)

print(f"Distritos coincidentes: {(~sin_barrio & ~discrepancias).sum():,}")
print(f"Distritos discrepantes: {discrepancias.sum():,}")
print(f"Estaciones sin barrio en el catálogo: {sin_barrio.sum():,}")

display(comparacion.loc[discrepancias | sin_barrio])

Distritos coincidentes: 536
Distritos discrepantes: 7
Estaciones sin barrio en el catálogo: 0


,estacion_id,nombre,codi_barri,codi_districte,districte_del_barri
12,13,"C/ MARINA, 25-33",67,1,10
15,17,"AV. MERIDIANA, 47 (D)",66,2,10
65,69,"CARRER DE LA MARINA,25-33",67,1,10
70,74,"AV. JOSEP TARRADELLAS, 133",19,2,4
225,235,"AV. PARAL.LEL, 98",11,2,3
292,304,"PG. MANUEL GIRONA, 7",23,4,5
383,395,"PL. CATALUNYA, 20",2,2,1


In [12]:
estaciones = tablas["estaciones"].copy()
distritos = tablas["distritos"]

if "codi_districte" in estaciones.columns:
    nombres_distrito = dict(zip(distritos["codi_districte"], distritos["districte"]))
    estaciones["districte_consultado"] = estaciones["codi_districte"].map(nombres_distrito)
elif "codi_barri" in estaciones.columns:
    barrio_a_distrito = dict(zip(tablas["barrios"]["codi_barri"], tablas["barrios"]["codi_districte"]))
    distrito_a_nombre = dict(zip(distritos["codi_districte"], distritos["districte"]))
    estaciones["districte_consultado"] = (
        estaciones["codi_barri"].map(barrio_a_distrito).map(distrito_a_nombre)
    )
elif "districte" in estaciones.columns:
    distritos_barcelona = set(distritos["districte"].dropna().str.strip())
    estaciones["districte_consultado"] = estaciones["districte"].where(
        estaciones["districte"].str.strip().isin(distritos_barcelona)
    )
else:
    estaciones["districte_consultado"] = pd.NA

sin_distrito = estaciones["districte_consultado"].isna()
print(f"Estaciones: {len(estaciones):,}")
print(f"¿Todas constan en un distrito del catálogo de Barcelona? {(~sin_distrito).all()}")
print(f"Sin distrito identificado: {sin_distrito.sum():,}")
display(estaciones["districte_consultado"].value_counts(dropna=False).rename("estaciones"))

for columna in ["altitud_m", "altitud", "capacidad"]:
    if columna in estaciones.columns:
        valores = pd.to_numeric(estaciones[columna], errors="coerce")
        print(f"\n{columna}:")
        display(valores.describe().to_frame())

Estaciones: 543
¿Todas constan en un distrito del catálogo de Barcelona? True
Sin distrito identificado: 0


districte_consultado
Eixample               120
Sant Martí              90
Ciutat Vella            63
Sants-Montjuïc          57
Sant Andreu             47
Horta-Guinardó          39
Sarrià-Sant Gervasi     38
Les Corts               37
Nou Barris              28
Gràcia                  24
Name: estaciones, dtype: int64


altitud_m:


,altitud_m
count,543.000000
mean,30.906077
std,84.047586
min,-999.000000
25%,9.000000
50%,25.000000
75%,54.000000
max,184.000000



capacidad:


,capacidad
count,543.000000
mean,27.064457
std,7.019838
min,0.000000
25%,24.000000
50%,27.000000
75%,30.000000
max,54.000000


In [4]:
viajes = tablas["viajes"]
print(f"¿viaje_id es único? {viajes['viaje_id'].is_unique}")
print(f"Filas con viaje_id duplicado: {viajes['viaje_id'].duplicated().sum()}")

¿viaje_id es único? False
Filas con viaje_id duplicado: 3088


In [8]:
viajes["tipo_bici"].value_counts(dropna=False)

tipo_bici
electrica     442910
ELECTRIC      145726
mecanica      140620
MECHANICAL     45725
Name: count, dtype: int64

In [9]:
duracion_min = (fin - inicio).dt.total_seconds() / 60

print(f"Duraciones nulas: {duracion_min.isna().sum():,}")
print(f"Duraciones negativas: {duracion_min.lt(0).sum():,}")
print(f"Viajes de 24 horas o más: {duracion_min.ge(24 * 60).sum():,}")

display(duracion_min.dropna().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).to_frame("duracion_min"))

px.histogram(
    duracion_min[duracion_min.between(0, 24 * 60)],
    nbins=60,
    title="Duración de los viajes (de 0 a 24 horas)",
    labels={"value": "Duración (minutos)", "count": "Número de viajes"},
)

Duraciones nulas: 86
Duraciones negativas: 14
Viajes de 24 horas o más: 1,853


,duracion_min
count,774895.000000
mean,22.732539
std,226.451386
min,-57.050000
25%,5.750000
50%,9.583333
75%,15.466667
90%,23.383333
99%,58.367667
max,7194.750000


**Ejemplo:** duración de los viajes en minutos

In [5]:
viajes = tablas["viajes"]
inicio = pd.to_datetime(viajes["fecha_hora_inicio"])
fin = pd.to_datetime(viajes["fecha_hora_fin"])
((fin - inicio).dt.total_seconds() / 60).describe()

count    774895.000000
mean         22.732539
std         226.451386
min         -57.050000
25%           5.750000
50%           9.583333
75%          15.466667
max        7194.750000
dtype: float64

**Ejemplo:** destinos que no están en el catálogo

In [6]:
catalogo = set(tablas["estaciones"]["estacion_id"])
viajes.loc[~viajes["estacion_destino"].isin(catalogo), "estacion_destino"].value_counts(dropna=False)

estacion_destino
9002.0    1559
9001.0    1553
NaN         86
Name: count, dtype: int64

In [7]:
# Vuestras comprobaciones aquí


In [10]:
for columna in ["estacion_origen", "estacion_destino"]:
    faltantes = viajes.loc[~viajes[columna].isin(catalogo), columna]
    print(f"{columna}: {len(faltantes):,} viajes con estación fuera del catálogo")
    display(faltantes.value_counts(dropna=False).rename("número de viajes"))

estacion_origen: 0 viajes con estación fuera del catálogo


Series([], Name: número de viajes, dtype: int64)

estacion_destino: 3,198 viajes con estación fuera del catálogo


estacion_destino
9002.0    1559
9001.0    1553
NaN         86
Name: número de viajes, dtype: int64

## 3. Una serie diaria: ¿algo raro?

Una serie por día es la forma más rápida de ver huecos y picos imposibles.

In [11]:
diario = viajes.groupby(inicio.dt.date).size().rename("viajes_muestra")
px.line(diario, title="Viajes de la muestra por día")